# Part 4: Data Analysis

In this notebook, we load the cleaned datasets generated by `1_data_preparation.ipynb` and run our analyses, visualizations, and statistical tests to answer our two research questions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Set seaborn style for nicer plots
sns.set_theme(style="whitegrid")

# Same order and colors for AI and Human in every plot
GROUP_ORDER = ['AI', 'Human']
GROUP_PALETTE = {'AI': '#4C72B0', 'Human': '#DD8452'}

# Load the cleaned datasets
rq1_df = pd.read_csv('../data/rq1_metrics.csv')
rq2_df = pd.read_csv('../data/rq2_metrics.csv')

print(f"Loaded RQ1 data: {rq1_df.shape}")
print(f"Loaded RQ2 data: {rq2_df.shape}")

## RQ1. Development Outcomes
**Question:** How do AI-related contributions differ from human contributions in terms of development outcomes?

**Approach**
- **Matched subset:** AI PRs from the same repositories and period as the Human PRs (see "Why the matched subset matters" in `1_data_preparation.ipynb`).
- **Human group = PRs opened by people** (`human_only` and `human_with_ai`). PRs opened by bots, by other AI agents or by AIDev agents that AIDev missed are left out (997 PRs).
- **Resolved PRs only** (`state == "closed"`, i.e. merged or closed without merging). Open PRs have no outcome yet.
- **Time to merge** and **number of commits** are measured on merged PRs only ("commits before acceptance").
- **Issue closing rate** and **time to close issue** only exist for PRs linked to an issue (about 20% of the PRs: 24% of AI PRs, 14% of Human PRs).
- **Tests:** chi-square for rates (effect size: Cramér's V), Mann-Whitney U for counts and durations, which are very skewed (effect size: rank-biserial r, negative = AI lower). p-values are corrected with Holm because we run 7 tests.
- **Reading effect sizes:** with ~12,000 PRs almost every difference is significant, so we judge differences on the effect size. Rough guide for V and |r|: < 0.1 negligible, 0.1–0.3 small, 0.3–0.5 medium, > 0.5 large.

In [ ]:
def chi_square(df, col):
    """Chi-square test of group x col. Returns the p-value and Cramér's V."""
    table = pd.crosstab(df['group'], df[col])
    chi2, p, _, _ = stats.chi2_contingency(table)
    cramers_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
    return p, cramers_v


def mann_whitney(df, col):
    """Mann-Whitney U test AI vs Human. Returns the p-value and the rank-biserial r (negative = AI lower)."""
    ai = df.loc[df['group'] == 'AI', col].dropna()
    human = df.loc[df['group'] == 'Human', col].dropna()
    u, p = stats.mannwhitneyu(ai, human)
    return p, 2 * u / (len(ai) * len(human)) - 1


def rq1_outcome_tests(resolved):
    """Runs the 7 RQ1 outcome tests on a set of resolved PRs and applies the Holm correction."""
    merged = resolved[resolved['is_merged']]
    with_issue = resolved[resolved['issue_is_closed'].notna()].copy()
    with_issue['issue_is_closed'] = with_issue['issue_is_closed'].astype(bool)

    # (metric, PRs it is measured on, column, test)
    tests = [
        ('Merge rate', resolved, 'is_merged', 'chi2'),
        ('Revised PR rate', resolved, 'is_revised', 'chi2'),
        ('Issue closing rate', with_issue, 'issue_is_closed', 'chi2'),
        ('Time to merge (hours)', merged, 'time_to_merge_hours', 'mwu'),
        ('Commits before merge', merged, 'num_commits', 'mwu'),
        ('Review rounds', resolved, 'num_review_rounds', 'mwu'),
        ('Time to close issue (hours)', with_issue, 'time_to_close_issue_hours', 'mwu'),
    ]

    rows = []
    for metric, data, col, test in tests:
        data = data.dropna(subset=[col])
        if test == 'chi2':
            values = data.groupby('group')[col].mean().reindex(GROUP_ORDER) * 100
            p, effect = chi_square(data, col)
            rows.append([metric, 'rate (%)', *values, 'chi-square', p, "Cramér's V", effect])
        else:
            values = data.groupby('group')[col].median().reindex(GROUP_ORDER)
            p, effect = mann_whitney(data, col)
            rows.append([metric, 'median', *values, 'Mann-Whitney U', p, 'rank-biserial r', effect])
        rows[-1][2:2] = data['group'].value_counts().reindex(GROUP_ORDER).tolist()

    results = pd.DataFrame(rows, columns=['metric', 'statistic', 'n_AI', 'n_Human', 'AI', 'Human',
                                          'test', 'p', 'effect_size', 'effect'])
    results['p_holm'] = multipletests(results['p'], method='holm')[1]
    return results


rq1_resolved = rq1_df[rq1_df['state'] == 'closed']
rq1_matched = rq1_resolved[rq1_resolved['in_matched_subset']]
rq1_merged = rq1_matched[rq1_matched['is_merged']]

print("Open PRs excluded:", rq1_df[rq1_df['in_matched_subset'] & (rq1_df['state'] == 'open')]['group'].value_counts().to_dict())
print("Resolved PRs (matched subset):", rq1_matched['group'].value_counts().to_dict())
print("Merged PRs (matched subset):", rq1_merged['group'].value_counts().to_dict())

### RQ1.1 Outcome comparison: AI vs Human

In [ ]:
rq1_results = rq1_outcome_tests(rq1_matched)

rq1_results.style.format({
    'AI': '{:.1f}', 'Human': '{:.1f}', 'p': '{:.2g}', 'p_holm': '{:.2g}', 'effect': '{:.3f}',
}).hide(axis='index')

In [ ]:
merge_rates = (rq1_matched.groupby('group')['is_merged'].mean().reindex(GROUP_ORDER) * 100).reset_index()

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=merge_rates, x='group', y='is_merged', hue='group',
                 order=GROUP_ORDER, palette=GROUP_PALETTE, legend=False)
plt.title('Merge Rate: AI vs Human (Matched Subset, Resolved PRs)')
plt.ylabel('Merge Rate (%)')
plt.xlabel('Developer Type')
plt.ylim(0, 100)

for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points')

plt.savefig('../figures/rq1_merge_rate.png', bbox_inches='tight')
plt.show()

In [ ]:
# Time to merge is very skewed (from seconds to months), so we use a log scale
print(rq1_merged.groupby('group')['time_to_merge_hours'].describe()[['count', '25%', '50%', '75%', 'mean']].round(2))

plt.figure(figsize=(8, 5))
sns.boxplot(data=rq1_merged, x='group', y='time_to_merge_hours', hue='group',
            order=GROUP_ORDER, palette=GROUP_PALETTE, legend=False)
plt.yscale('log')
plt.title('Time to Merge: AI vs Human (Matched Subset, Merged PRs)')
plt.ylabel('Time to Merge (hours) - Log Scale')
plt.xlabel('Developer Type')

plt.savefig('../figures/rq1_time_to_merge_boxplot.png', bbox_inches='tight')
plt.show()

### RQ1.2 Merge rate by task category

AI PRs contain more bug fixes and documentation, and Human PRs more CI/CD and maintenance work (see RQ2.2). If the merge-rate gap only came from the kind of task, it would disappear within each category.

In [ ]:
category_order = ['All'] + list(rq1_matched['task_category'].value_counts().index)
category_rates = pd.concat([
    rq1_matched.assign(task_category='All'),
    rq1_matched,
]).groupby(['task_category', 'group'])['is_merged'].mean().mul(100).reset_index()

plt.figure(figsize=(9, 6))
ax = sns.barplot(data=category_rates, y='task_category', x='is_merged', hue='group',
                 order=category_order, hue_order=GROUP_ORDER, palette=GROUP_PALETTE)
plt.title('Merge Rate by Task Category (Matched Subset, Resolved PRs)')
plt.xlabel('Merge Rate (%)')
plt.ylabel('Task Category')
plt.xlim(0, 100)
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', padding=3)
plt.legend(title='Developer Type', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.savefig('../figures/rq1_merge_rate_by_category.png', bbox_inches='tight')
plt.show()

# Chi-square test within each category (Holm correction over the 7 categories;
# the "All" row is the overall test from RQ1.1, so it is not part of this correction)
rows = []
for category in category_order:
    subset = rq1_matched if category == 'All' else rq1_matched[rq1_matched['task_category'] == category]
    p, v = chi_square(subset, 'is_merged')
    rates = subset.groupby('group')['is_merged'].mean().reindex(GROUP_ORDER) * 100
    counts = subset['group'].value_counts().reindex(GROUP_ORDER)
    rows.append([category, counts['AI'], counts['Human'], rates['AI'], rates['Human'], p, v])

category_tests = pd.DataFrame(rows, columns=['task_category', 'n_AI', 'n_Human', 'AI (%)', 'Human (%)', 'p', "Cramér's V"])
is_category = category_tests['task_category'] != 'All'
category_tests.loc[is_category, 'p_holm'] = multipletests(category_tests.loc[is_category, 'p'], method='holm')[1]
category_tests.style.format({
    'AI (%)': '{:.1f}', 'Human (%)': '{:.1f}', 'p': '{:.2g}', 'p_holm': '{:.2g}', "Cramér's V": '{:.3f}',
}, na_rep='—').hide(axis='index')

### RQ1.3 Logistic regression: does being an AI PR still predict merging?

AI PRs could be merged less often simply because they contain different tasks or bigger changes. To rule this out, we predict `is_merged` from the group while **controlling** for the task category and the size of the PR (lines and files changed, log-transformed because they are very skewed). The result is the effect of being an AI PR for PRs of the same task and size.

PRs from the same repository are not independent (same maintainers, same rules), so standard errors are clustered by repository.

An **odds ratio** below 1 means lower odds of being merged than Human PRs.

In [ ]:
model_df = rq1_matched.merge(rq2_df[['id', 'code_churn', 'num_files_changed']], on='id')
model_df['merged'] = model_df['is_merged'].astype(int)
model_df['is_ai'] = (model_df['group'] == 'AI').astype(int)
model_df['log_churn'] = np.log1p(model_df['code_churn'])
model_df['log_files'] = np.log1p(model_df['num_files_changed'])

merge_model = smf.logit(
    "merged ~ is_ai + C(task_category, Treatment('Bug fixes')) + log_churn + log_files",
    data=model_df,
).fit(disp=0, cov_type='cluster', cov_kwds={'groups': model_df['repo_id']})

# Only the AI vs Human row answers RQ1; task category and size are controls
ci_low, ci_high = np.exp(merge_model.conf_int().loc['is_ai'])
p = merge_model.pvalues['is_ai']
ai_effect = pd.DataFrame({
    'Factor': ['AI PR (vs Human)'],
    'Odds ratio': [f"{np.exp(merge_model.params['is_ai']):.2f}"],
    '95% CI': [f'{ci_low:.2f}–{ci_high:.2f}'],
    'p': ['< 0.001' if p < 0.001 else f'{p:.2f}'],
})

print(f"PRs: {int(merge_model.nobs)}, repositories: {model_df['repo_id'].nunique()}, pseudo R²: {merge_model.prsquared:.3f}")
print("Controlled for: task category and PR size (lines and files changed)")
ai_effect.style.hide(axis='index')

### RQ1 Answer

**AI PRs are merged much less often than Human PRs, but the ones that are merged go through a similar process.**

- **Merge rate:** 59.5% for AI vs 85.1% for Human (chi-square p < 0.001, Cramér's V = 0.28, small-to-medium). This is the clearest difference in RQ1.
- **Not explained by task type or size:** the gap exists in every task category (largest for bug fixes: 56% vs 88%, smallest for documentation: 80% vs 88%). In the logistic regression, AI PRs have about 0.25 times the odds of being merged as Human PRs with the same task category and size (95% CI 0.21–0.31).
- **Merged PRs look similar:** AI PRs are merged a bit faster (median 2.0 h vs 4.0 h, significant but negligible, r = -0.09) and have the same number of commits (median 2 for both, not significant).
- **More human review on AI PRs:** 34% of AI PRs and 21% of Human PRs get at least one human review (small effect, r = 0.12, AI higher). About 41–43% are revised in both groups (not significant).
- **Linked issues:** issue closing rates are close (83.7% vs 87.5%, not significant after Holm correction). Issues linked to AI PRs are closed faster (median 218 h vs 402 h, small effect r = -0.12). Only about 20% of PRs are linked to an issue (24% of AI PRs, 14% of Human PRs), so these results are less reliable.

**Why the matched subset:** on all AI PRs, the merge rate is about 77% (see "Why the matched subset matters" in `1_data_preparation.ipynb`). That number is pushed up by repositories with no Human PRs (e.g. one personal repository holds 27% of all AI PRs, 87% of them merged). Comparing AI and Human PRs from the same repositories and period gives the fairer picture.

**Limitations:** `is_revised` counts any reopening or review request, and some agents request a review automatically, so it is a rough measure of revision. `num_commits` is capped at 30 by the data source (about 2% of PRs).

## RQ2. Contribution Characteristics
**Question:** How do AI-related and human contributions differ in terms of size, scope, and type?

**Approach**
- **Matched subset**, all PRs (open ones included: the size and type of a PR do not depend on whether it was merged).
- **Size:** lines added, lines deleted and code churn (added + deleted). **Scope:** number of files changed. **Type:** task category and the kind of files touched (docs, tests, config).
- PRs without commit details (`num_files_changed == 0`, mostly AI PRs) are excluded from the size, scope and file-type analyses: their 0 means "missing", not "empty".
- **Tests:** Mann-Whitney U for size and scope (effect size: rank-biserial r, negative = AI lower), chi-square for type (effect size: Cramér's V). p-values are corrected with Holm within each analysis. Effect sizes are read with the same guide as RQ1.

### RQ2.1 Size and scope

In [ ]:
rq2_matched = rq2_df[rq2_df['in_matched_subset']]
# PRs without commit details have 0 files: their size is missing, not zero
rq2_with_files = rq2_matched[rq2_matched['num_files_changed'] > 0]

print("PRs (matched subset):", rq2_matched['group'].value_counts().to_dict())
print("PRs with commit details:", rq2_with_files['group'].value_counts().to_dict())

size_metrics = {
    'additions': 'Lines added',
    'deletions': 'Lines deleted',
    'code_churn': 'Code churn (added + deleted)',
    'num_files_changed': 'Files changed',
}

rows = []
for col, metric in size_metrics.items():
    quartiles = rq2_with_files.groupby('group')[col].quantile([0.25, 0.5, 0.75]).unstack().reindex(GROUP_ORDER)
    p, r = mann_whitney(rq2_with_files, col)
    rows.append([metric, *quartiles.loc['AI'], *quartiles.loc['Human'], p, r])

size_results = pd.DataFrame(rows, columns=['metric', 'AI Q1', 'AI median', 'AI Q3',
                                           'Human Q1', 'Human median', 'Human Q3', 'p', 'rank-biserial r'])
size_results['p_holm'] = multipletests(size_results['p'], method='holm')[1]
size_results.style.format({
    'AI Q1': '{:.0f}', 'AI median': '{:.0f}', 'AI Q3': '{:.0f}',
    'Human Q1': '{:.0f}', 'Human median': '{:.0f}', 'Human Q3': '{:.0f}',
    'p': '{:.2g}', 'rank-biserial r': '{:.3f}', 'p_holm': '{:.2g}',
}).hide(axis='index')

In [ ]:
# Boxplot for Code Churn (log scale because code churn can be very large)
plt.figure(figsize=(8, 5))
sns.boxplot(data=rq2_with_files, x='group', y='code_churn', hue='group',
            order=GROUP_ORDER, palette=GROUP_PALETTE, legend=False)
plt.yscale('log')
plt.title('Code Churn per PR: AI vs Human (Matched Subset)')
plt.ylabel('Code Churn (Lines added + deleted) - Log Scale')
plt.xlabel('Developer Type')

plt.savefig('../figures/rq2_code_churn_boxplot.png', bbox_inches='tight')
plt.show()

### RQ2.2 Type

Two views of the type of contribution: the **task category** of the PR (what kind of work) and the **kind of files** it touches (docs, tests, config).

In [ ]:
# Share of each group's PRs in each task category (each group sums to 100%)
task_shares = (rq2_matched.groupby('group')['task_category'].value_counts(normalize=True)
               .mul(100).rename('share').reset_index())
task_order = rq2_matched['task_category'].value_counts().index

plt.figure(figsize=(9, 6))
ax = sns.barplot(data=task_shares, y='task_category', x='share', hue='group',
                 order=task_order, hue_order=GROUP_ORDER, palette=GROUP_PALETTE)
plt.title('Task Category Mix: AI vs Human (Matched Subset)')
plt.xlabel("Share of the group's PRs (%)")
plt.ylabel('Task Category')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', padding=3)
plt.legend(title='Developer Type', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.savefig('../figures/rq2_task_category_mix.png', bbox_inches='tight')
plt.show()

p, v = chi_square(rq2_matched, 'task_category')
print(f"Task category vs group: chi-square p = {p:.2g}, Cramér's V = {v:.3f}")

In [ ]:
# Share of PRs touching at least one file of each kind (a PR can touch several kinds)
file_types = {
    'is_docs_change': 'Docs files',
    'is_test_change': 'Test files',
    'is_config_change': 'Config files',
}

rows = []
for col, kind in file_types.items():
    rates = rq2_with_files.groupby('group')[col].mean().reindex(GROUP_ORDER) * 100
    p, v = chi_square(rq2_with_files, col)
    rows.append([kind, rates['AI'], rates['Human'], p, v])

file_type_results = pd.DataFrame(rows, columns=['PRs touching', 'AI (%)', 'Human (%)', 'p', "Cramér's V"])
file_type_results['p_holm'] = multipletests(file_type_results['p'], method='holm')[1]
file_type_results.style.format({
    'AI (%)': '{:.1f}', 'Human (%)': '{:.1f}', 'p': '{:.2g}', "Cramér's V": '{:.3f}', 'p_holm': '{:.2g}',
}).hide(axis='index')

### RQ2 Answer

**AI and Human PRs have about the same size and scope, but they do somewhat different kinds of work: AI PRs do less CI/CD, configuration and maintenance work.**

- **Size and scope are similar:** median code churn is 116 lines for AI vs 123 for Human, and median files changed is 3 vs 4. Lines added and code churn do not differ significantly; lines deleted and files changed are significant but negligible (|r| < 0.1). AI PRs are more uniform in size: fewer very small PRs (Q1 churn 26 vs 16 lines) and far fewer very large ones (Q3 537 vs 1,016 lines).
- **Different task mix** (chi-square p < 0.001, Cramér's V = 0.16, small): Human PRs contain more CI/CD or configuration work (7% vs 3%) and maintenance (11% vs 5%). AI PRs contain relatively more bug fixes (33% vs 29%), documentation (12% vs 9%) and testing (5% vs 3%).
- **Different files touched:** fewer AI PRs touch config files (31% vs 42%, small effect V = 0.11). AI PRs touch test files a bit more often (45% vs 41%) and docs files a bit less often (33% vs 38%), both negligible effects.

**Link with RQ1:** these differences in size and task type do not explain the lower merge rate of AI PRs (see RQ1.3).

**Limitations:** the docs/test/config flags come from file names (e.g. "test" anywhere in the path), so they are approximate. The task category comes from the AIDev dataset's automatic classification. About 2% of AI PRs have no commit details and are excluded from the size, scope and file-type analyses.